# E4 -- two-component coupled quartic chain (24D): plot notebook

**This notebook only reads saved results.**

It must not call a sampler, a PT tuner, an LSC quadrature refinement, a `dt` refinement, or a reference builder, and it must not recompute any official metric. Every number drawn here already exists in a run's `metrics_timeseries.csv` or `cost_timeseries.csv`, written by `E4_coupled_quartic_chain_run.ipynb` at run time.

Scatter, CDF, histogram, and KDE panels are **display only**. They visualise the saved sample snapshots and **never override, correct, or stand in for** the numbers in `metrics_timeseries.csv`. If a picture and a saved metric disagree, the saved metric is the result.

Method colours, markers, and display names come from `configs/registry.yaml`; which runs to draw and how to lay them out comes from `configs/plots/manuscript.yaml`. Neither table is redefined here.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, "..")  # importable when launched from notebooks/

from src.catalog import select_runs
from src.plotting import (curve_figure, load_plot_config, load_runs,
                          save_figure, snapshot_figure)

REPO_ROOT = Path("..")
EXPERIMENT_ID = "E4"

plot_config = load_plot_config(REPO_ROOT / "configs" / "plots" / "manuscript.yaml")
defaults = plot_config["defaults"]
spec = plot_config[EXPERIMENT_ID]
figures = spec["figures"]

EXPERIMENT_DIR = REPO_ROOT / "results" / spec["experiment_key"]
OUTPUT_DIR = REPO_ROOT / defaults["output_root"] / spec["experiment_key"]
FORMATS = tuple(defaults["output_formats"])  # png, pdf, svg, tiff
FIGURES = {}


def load_available(experiment_dir, spec, **kwargs):
    """Load the spec's runs, tolerating methods this campaign could not run.

    Canonical (untamed) variants are expected to be unusable on several of
    these targets: the drift is not truncated, so at the step size the run
    stage settled on they are genuinely unstable. That is a result, not a plotting problem, and a
    figure must still draw the methods that did run. Uncalibratable methods are
    annotated by the plotting module; anything still missing here is dropped
    with a printed warning rather than aborting the notebook.
    """
    try:
        return load_runs(experiment_dir, spec, **kwargs)
    except ValueError as error:
        if "requires missing methods" not in str(error):
            raise
        print(f"warning: {error}")
        print("plotting only the runs that completed")
        # methods=None disables the all-methods-required check; the spec's
        # variant filters still exclude unrelated methods.
        return load_runs(experiment_dir, spec, methods=None, **kwargs)


print(f"{EXPERIMENT_ID}: {len(figures)} specified figures -> {OUTPUT_DIR}")

E4: 4 specified figures -> ../figures/E4_coupled_quartic_chain


## What is plottable

Load the derived catalog (rebuilding it from the manifests if it is missing) and list the runs it admits, so it is visible up front which methods, variants, and step sizes this notebook can actually draw. Nothing is run here; this is a directory listing.

In [2]:
runs = select_runs(EXPERIMENT_DIR, latest_only=defaults["latest_run_only"])

print(f"{len(runs)} plottable runs\n")
print(f"{'method':<12}{'variant label':<34}{'tame':<7}{'dt':<10}run id")
for row in runs:
    print(f"{row['method']:<12}{row['variant_label']:<34}"
          f"{str(row['tame']):<7}{str(row['dt']):<10}{row['run_id']}")

17 plottable runs

method      variant label                     tame   dt        run id
FLA         FLA alpha=1.6, canonical          False  0.001     FLA-alpha1.6-canonical-dt0.001-20260806T222326433619Z
FLA         FLA alpha=1.6, tamed              True   0.0005    FLA-alpha1.6-tamed-dt0.0005-20260806T222424452663Z
FLA         FLA alpha=1.7, canonical          False  0.0005    FLA-alpha1.7-canonical-dt0.0005-20260806T222514042225Z
FLA         FLA alpha=1.7, tamed              True   0.0005    FLA-alpha1.7-tamed-dt0.0005-20260806T222608398207Z
FLA         FLA alpha=1.8, canonical          False  0.0005    FLA-alpha1.8-canonical-dt0.0005-20260806T222658651665Z
FLA         FLA alpha=1.8, tamed              True   0.0005    FLA-alpha1.8-tamed-dt0.0005-20260806T222753102296Z
LSC-CP      LSC-CP, tamed                     True   0.002     LSC-CP-tamed-dt0.002-20260806T223910569248Z
LSC-CP-RA   LSC-CP-RA (A=4), tamed            True   0.002     LSC-CP-RA-A4-tamed-dt0.002-20260806T2246151216

### Figure E4.1 -- order-parameter samples

Reference order-parameter contours with each method's $(m_x, m_y)$ samples at one matched simulation time, on shared axes and shared contour levels.

The background is a **reference estimate** built from the numerical PT-MALA reference samples. It is **not** an exact surface, and the caption must say estimate.

In [3]:
figure_spec = figures["E4.1_order_parameter_scatter"]

FIGURES["E4.1_order_parameter_scatter"] = snapshot_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E4.1_order_parameter_scatter"]

<Figure size 720x540 with 6 Axes>

### Figure E4.2 -- physical correctness

Five panels, each drawn against simulation time and against FEE:

1. phase-weight Jensen-Shannon divergence;
2. order-parameter SW$_2$;
3. order-parameter MMD$^2$;
4. energy-per-site absolute error;
5. relative Frobenius error of the susceptibility matrix.

In [4]:
figure_spec = figures["E4.2_physical_correctness"]

FIGURES["E4.2_physical_correctness"] = curve_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E4.2_physical_correctness"]

<Figure size 740x1190 with 10 Axes>

### Figure E4.3 -- supplement diagnostics

The diagnostic panels listed under `E4.3_supplement` in the plot specification: order-parameter contour scatter, $m_x$ and $m_y$ marginal CDFs, the $\|m\|$ radial distribution, phase-occupancy bars, energy-per-site and coherence CDFs, the kink-density histogram, the two-point correlation profile, the susceptibility heat map, heat-capacity and Binder-cumulant bars, and the reference validation summary.

These are **static equilibrium diagnostics**. There are no first-passage, transition-count, round-trip, relay-path, or kinetic transition-matrix figures, and none of these panels overrides a number in `metrics_timeseries.csv`.

In [5]:
figure_spec = figures["E4.3_supplement"]

FIGURES["E4.3_supplement"] = snapshot_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E4.3_supplement"]

<Figure size 930x1130 with 13 Axes>

### Figure E4.4 -- LSC score potential-evaluation cost

The LSC-only cost figure: full LSC-CP against LSC-CP-RA(A). The x axis counts LSC score potential evaluations only and is not a complete computational cost.

In [6]:
figure_spec = figures["E4.4_lsc_score_cost"]

FIGURES["E4.4_lsc_score_cost"] = curve_figure(
    load_available(EXPERIMENT_DIR, figure_spec), figure_spec)
FIGURES["E4.4_lsc_score_cost"]

<Figure size 400x530 with 2 Axes>

## Canonical, tamed, and paired views

The main curve figure is regenerated three times, from the same saved runs: **canonical only**, **tamed only**, and the **paired** canonical-versus-tamed overlay.

The convention, taken from `configs/registry.yaml` and the plot defaults:

* the **method** sets the **colour**, and taming never changes it;
* **canonical** is a **solid** line;
* **tamed** is a **dashed** line;
* **hyperparameter values** are distinguished by **marker**, and the value is written into the legend label.

So colour answers "which method", line style answers "tamed or not", and marker answers "which hyperparameter value".

In [7]:
MAIN_CURVE_FIGURE = "E4.2_physical_correctness"

for view in defaults["tame_views"]:
    view_spec = {**figures[MAIN_CURVE_FIGURE], "tame_view": view}
    FIGURES[f"{MAIN_CURVE_FIGURE}__{view}"] = curve_figure(
        load_available(EXPERIMENT_DIR, view_spec), view_spec)

print("tame views:", list(defaults["tame_views"]))

tame views: ['canonical_only', 'tamed_only', 'paired']


## Export

Every figure built above is written to **PNG, PDF, SVG, and TIFF** under `figures/<experiment key>/`, from the format list in the plot defaults. Re-running this cell overwrites the files in place; it never touches anything under `results/`.

In [8]:
for name, figure in FIGURES.items():
    save_figure(figure, name, OUTPUT_DIR, formats=FORMATS)
    print(f"saved {name}  [{', '.join(FORMATS)}]")

print(f"\n{len(FIGURES)} figures written under {OUTPUT_DIR}")

saved E4.1_order_parameter_scatter  [png, pdf, svg, tiff]


saved E4.2_physical_correctness  [png, pdf, svg, tiff]


saved E4.3_supplement  [png, pdf, svg, tiff]


saved E4.4_lsc_score_cost  [png, pdf, svg, tiff]


saved E4.2_physical_correctness__canonical_only  [png, pdf, svg, tiff]


saved E4.2_physical_correctness__tamed_only  [png, pdf, svg, tiff]


saved E4.2_physical_correctness__paired  [png, pdf, svg, tiff]

7 figures written under ../figures/E4_coupled_quartic_chain
